# Practical 11 — Text Classification (Naive Bayes & SVM)

**Course:** NLP
**Name:**  <!-- fill in -->
**Date:**  <!-- fill in -->

## Aim
To train Naive Bayes and SVM classifiers on TF-IDF features to predict review sentiment, and to test whether a 15-example labeled dataset is actually large enough to evaluate a classifier's real-world accuracy - continuing the same data-scale question from Practicals 8 and 10, this time for supervised learning.

## Theory

**Naive Bayes** and **SVM** are both standard supervised classifiers for text, typically fed TF-IDF or Bag-of-Words features. Naive Bayes assumes feature independence and models class-conditional word probabilities; SVM finds a decision boundary that maximizes the margin between classes.

Both need **labeled** data - unlike every previous practical, this one requires a ground-truth sentiment label for each review, which our dataset doesn't come with. Step 1 creates that.

The real point of this practical, though, is a different data-scale problem than Practicals 8 and 10. Even if a classifier trains "successfully" on 15 examples, there's a separate question: **can you trust an accuracy number computed from a test set that small?** A standard 70/30 train/test split on 15 examples leaves only 4-5 test examples - meaning a single example changes the reported accuracy by roughly 20-25 percentage points. This practical tests that directly, rather than just asserting it.

## Algorithm

1. Assign a ground-truth sentiment label to each of the 15 reviews (starting from a suggested draft, checked against your own reading - particularly the ambiguous ones).
2. Build TF-IDF features (same approach as Practical 7) and train Naive Bayes + SVM on **all 15** labeled examples, evaluating on that same data.
3. Do a real train/test split (70/30) and evaluate on the held-out 30% instead.
4. Repeat the train/test split 10 times with different random seeds and look at how much the resulting accuracy actually varies.

In [11]:
import sys, os
sys.path.append(os.path.abspath("../python"))

import pandas as pd
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC

import vectorizers
import classifiers as clf

pd.set_option("display.max_colwidth", None)

df = pd.read_csv("../datasets/sample_reviews.csv")
print(f"Loaded {len(df)} reviews")


Loaded 15 reviews


### Step 1 — Ground-truth sentiment labels

**This is a starting draft based on one reading of the reviews, not settled fact - go through the list yourself against the actual review text below, and edit anything you disagree with. Reviews 9 and 13 in particular are genuinely ambiguous/mixed (Practical 9 already found this for review 9) - your call on those two matters more than the others.**

| id | review (shortened) | draft label |
|----|---|---|
| 1 | "ABSOLUTELY fantastic" | positive |
| 2 | "Worst film... don't waste" | negative |
| 3 | "solid 7/10, great visuals, but dragged" | positive |
| 4 | "good acting... outdid himself" | positive |
| 5 | "meh, nothing special, wouldn't watch again" | negative |
| 6 | "by far his best work, must watch" | positive |
| 7 | "pure boredom, would not recommend" | negative |
| 8 | "Best film... 10/10" | positive |
| 9 | "not great, not terrible, middle" | **ambiguous - your call** |
| 10 | "worth every penny" | positive |
| 11 | "fell asleep, slow pacing" | negative |
| 12 | "masterpiece... best films of the decade" | positive |
| 13 | "not BAD, but expected way more" | **ambiguous - your call** |
| 14 | "5 stars... worth the price" | positive |
| 15 | "Confusing plot, weak dialogue" | negative |


In [12]:
# Edit this list directly if you disagree with any label above -
# it must be in the same order as the CSV rows (review id 1 through 15).
labels = [
    "positive",  # 1
    "negative",  # 2
    "positive",  # 3
    "positive",  # 4
    "negative",  # 5
    "positive",  # 6
    "negative",  # 7
    "positive",  # 8
    "negative",  # 9  <- ambiguous, reconsider this one yourself
    "positive",  # 10
    "negative",  # 11
    "positive",  # 12
    "negative",  # 13 <- ambiguous, reconsider this one yourself
    "positive",  # 14
    "negative",  # 15
]

assert len(labels) == len(df), "labels list must have exactly one entry per review"
df["label"] = labels
df[["id", "review", "label"]]


,id,review,label
0,1,This movie was ABSOLUTELY fantastic!!! I've never seen anything like it before.,positive
1,2,"Worst film of 2026. Don't waste your $12 on a ticket, I promise you won't like it.",negative
2,3,"A solid 7/10 - great visuals, but the plot dragged on for way too long...",positive
3,4,I can't believe how good the acting was!! Robert De Niro really outdid himself this time.,positive
4,5,"meh. it was fine i guess?? nothing special, wouldn't watch again tbh",negative
5,6,The director's 3rd project is by FAR his best work -- a must watch in theaters.,positive
6,7,Two hours and 15 mins of pure boredom. 2/10 would not recommend to anyone.,negative
7,8,"WOW!!! Best film I've seen in years, hands down. 10/10 no notes.",positive
8,9,"It's okay... not great, not terrible. Somewhere in the middle I'd say.",negative
9,10,Ticket prices are insane these days ($18.50!) but this one was worth every penny.,positive


### Step 2 — Train and evaluate on the SAME 15 examples (in-sample)

In [13]:
tfidf_matrix, feature_names = vectorizers.tfidf_vectorize(df["review"].tolist())
y = df["label"].tolist()

nb_model = clf.train_naive_bayes(tfidf_matrix, y)
svm_model = clf.train_svm(tfidf_matrix, y)

nb_result = clf.evaluate_classifier(nb_model, tfidf_matrix, y)
svm_result = clf.evaluate_classifier(svm_model, tfidf_matrix, y)

print(f"Naive Bayes in-sample accuracy: {nb_result['accuracy']:.2%}")
print(f"SVM in-sample accuracy:         {svm_result['accuracy']:.2%}")


Naive Bayes in-sample accuracy: 100.00%
SVM in-sample accuracy:         100.00%


**If either number came back at or near 100%, that is NOT evidence the model actually learned to detect sentiment - it has 15 labeled examples and a TF-IDF feature space with well over 100 dimensions, so it very plausibly just memorized the training data outright. This is the same lesson as overfitting with too few examples relative to feature count - keep this in mind for Step 3.**

### Step 3 — A real train/test split

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    tfidf_matrix, y, test_size=0.3, random_state=42
)

print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

nb_split = clf.train_naive_bayes(X_train, y_train)
svm_split = clf.train_svm(X_train, y_train)

nb_test_result = clf.evaluate_classifier(nb_split, X_test, y_test)
svm_test_result = clf.evaluate_classifier(svm_split, X_test, y_test)

print(f"Naive Bayes TEST accuracy: {nb_test_result['accuracy']:.2%}")
print(f"SVM TEST accuracy:         {svm_test_result['accuracy']:.2%}")


Train size: 10, Test size: 5
Naive Bayes TEST accuracy: 0.00%
SVM TEST accuracy:         0.00%


### Step 4 — How much does accuracy swing across different splits?

In [15]:
nb_accuracies = clf.repeated_holdout_variance(tfidf_matrix, y, lambda: MultinomialNB(), test_size=0.3, n_repeats=10)
svm_accuracies = clf.repeated_holdout_variance(tfidf_matrix, y, lambda: SVC(kernel="linear"), test_size=0.3, n_repeats=10)

print("Naive Bayes accuracy across 10 different random splits:")
print([f"{a:.0%}" for a in nb_accuracies])
print(clf.summarize_variance(nb_accuracies))
print()
print("SVM accuracy across 10 different random splits:")
print([f"{a:.0%}" for a in svm_accuracies])
print(clf.summarize_variance(svm_accuracies))


Naive Bayes accuracy across 10 different random splits:
['20%', '80%', '20%', '20%', '40%', '80%', '20%', '40%', '60%', '20%']
{'mean': 0.4, 'std': 0.23664319132398465, 'min': 0.2, 'max': 0.8}

SVM accuracy across 10 different random splits:
['20%', '80%', '20%', '20%', '40%', '80%', '20%', '40%', '60%', '20%']
{'mean': 0.4, 'std': 0.23664319132398465, 'min': 0.2, 'max': 0.8}


---
## Output

*Run every cell above top to bottom, then paste or describe your actual output here — your final label list (note anything you changed from the draft and why), the in-sample accuracies, the single train/test split accuracies, and the full spread of 10 accuracy values (with mean/std/min/max) for both classifiers. Don't fill this in until you've actually run it.*


## Observations & Conclusion

Answer these based on what you actually saw when you ran the notebook:

- Did you change either of the two ambiguous labels (reviews 9, 13) from the draft? What was your reasoning?
- What were the in-sample accuracies in Step 2 - did they come back suspiciously high, and does that actually tell you the model generalizes?
- How different was the single train/test split accuracy (Step 3) from the in-sample accuracy (Step 2)?
- Looking at the full spread of 10 accuracy values in Step 4 (not just the mean) - how much did accuracy vary just from changing which examples landed in the test set? What does that tell you about trusting a single accuracy number from a dataset this size?
- Pulling this together with Practicals 8 and 10: is this the same "not enough data" problem in a new form, or a genuinely different issue specific to supervised evaluation?

*(Write 4-6 sentences here in your own words once you've run the notebook.)*


---
## Viva Prep — Practice Questions

1. **Why can't accuracy on the training set itself be trusted as a measure of a classifier's real-world performance?**
   A model can achieve high training accuracy simply by memorizing the specific examples it was trained on, especially when the feature space is large relative to the number of examples - this says nothing about how it will perform on new, unseen data.

2. **Why is a 70/30 train/test split especially unreliable on a 15-example dataset?**
   It leaves only 4-5 test examples, so each individual prediction changes the reported accuracy by roughly 20-25 percentage points - a single hard or easy example can swing the number dramatically, making any one accuracy figure close to meaningless.

3. **What does a wide spread of accuracy values across repeated random splits actually demonstrate?**
   That the reported accuracy is highly dependent on which specific examples happened to land in the test set, rather than reflecting a stable, reliable estimate of the model's true performance - the variance itself is evidence the number can't be trusted in isolation.

4. **What's the practical difference between how Naive Bayes and SVM make predictions?**
   Naive Bayes estimates class-conditional probabilities for each word under an independence assumption and picks the most probable class; SVM instead finds a decision boundary (hyperplane) that maximizes the margin between classes in feature space, without assuming feature independence.

5. **How does this practical's finding relate to Practical 8 (Word2Vec) and Practical 10 (LDA)?**
   All three show a version of the same underlying issue - certain NLP techniques (embeddings, topic models, and now supervised evaluation) require enough data for their statistics to stabilize; with too little, the outputs (similarity scores, topic assignments, accuracy numbers) can look plausible on the surface while actually being closer to noise.
